# 31 · Join a table to the study area, checked against a pandas merge

## What this checks

**Join a table to the study area** gives each part of the study area the value from the table row with its name. This
check covers release 0.7.0.

Snow's Table VIII gives 32 sub-districts a population and a death count. The study area has **33** named parts,
because Streatham is drawn as two polygons. Does each part get the rate that belongs to its own name? And does the
join still refuse a name it should not accept?

This is the step that turns a table of rates into a map of rates, a choropleth. The South London example argues for
that map in words. The widget shipped on 25 October 2026 with no independent evidence.

## How it is checked

Fieldwork matches names after folding them: it ignores punctuation and capitals. We do **not** write that folding
again. A second copy by the same author would only be our own arithmetic checking our own. So three separate things
are checked, each independent in its own way:

- **The join, given the folded names.** We use **pandas**, Python's table library, by other authors. Its `merge` keeps
  every part and adds the matching row. This is fully independent.
- **Plain matching, without folding.** Exact matching of the names as written needs no folding at all. So it is as
  independent as anything here can be.
- **What folding is allowed to do.** This is a rule the folding must obey, not a second copy. Every folded name must
  equal the name with every character that is not a letter or digit deleted. And no two different names may fold to
  the same thing. That is weaker than an independent fold. But it still refuses a fold that turned `St.` into `Saint`,
  dropped a word, merged two names, or changed a letter.

**A negative control, because the data cannot supply one.** In Snow's table and these boundaries, no pair of names
differs by more than punctuation. So nothing in the example tests a name that *ought not* to match. A rule that
matched too easily would also score 32 of 32. So we rename one part to `Camberwell Green`, a real place the table does
not carry. The join must report it, not find it a home.

For that one case we also run **rapidfuzz** and **jellyfish**, two libraries that score how alike two spellings are.
They say how close `Camberwell Green` is to `Camberwell`, and so how close a matcher that accepts similar names would
come to accepting it. This widget matches names exactly, with no threshold, so that score is what its choice is worth.
We report the score; we do not require anything of it.

## What should happen

1. **Every part's value** equals a pandas join on the folded name, compared part by part, empty values included. The
   two Streatham parts carry one value between them. They have the same id, so they are one unit, and they are not
   flagged as sharing a row. A name is shared only where it names more than one unit.
2. **The counts** all agree: parts, rows, matched parts, matched rows, unmatched and shared.
3. **Plain matching:** 23 of 32 rows match without folding. The 9 that do not are exactly the 9 Fieldwork reports.
   Each differs from its row only in characters that are not letters or digits.
4. **The folding changed nothing else:** every folded name obeys the rule above, and no two different names fold
   together.
5. **The negative control** is reported unmatched and named, and the other 32 parts are unchanged.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports, requested one at a time, so that a package the browser kernel lacks does not stop
# the others from loading. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "rapidfuzz", "jellyfish"]
try:
    import pyodide_js
except ImportError:
    pyodide_js = None
    print("Not a Pyodide kernel; using the ambient environment.")
if pyodide_js:
    for package in NEEDED:
        try:
            await pyodide_js.loadPackage(package)
            print("Loaded:", package)
        except Exception:
            print("Not in this kernel:", package)

In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_31_area_join import run

report = run(Path.cwd() / "fixtures" / "area-join-snow-001.json")
print("agrees:", report["agrees"])
for c in report["comparisons"]:
    print(f"\n{c['note']}   [{c['normalisation']}]")
    print(f"  parts matched   Fieldwork {c['fieldwork']['matchedParts']:>2}   pandas {c['pandas']['matchedParts']:>2}")
    print(f"  rows matched              {c['fieldwork']['matchedRows']:>2}          {c['pandas']['matchedRows']:>2}")
    print(f"  shared parts              {c['fieldwork']['sharedParts']:>2}          {c['pandas']['sharedParts']:>2}")
    if c["unmatchedPartNames"]:
        print(f"  unmatched: {', '.join(c['unmatchedPartNames'])}")
    print(f"  diagnostics: {', '.join(c['diagnosticCodes'])}")
    for name, ok in c["checks"].items():
        print(f"    {'OK ' if ok else 'DIFF'} {name}")


## The result on record

First recorded on 27 October 2026, against release 0.1.0. Rerun on 26 November 2026, against release 0.7.0:
**Agrees**.

Three cases over Snow's Table VIII, using the declared rate:

- **With folding:** 33 of 33 parts and 32 of 32 rows matched. None is shared: Streatham's two polygons are one unit,
  and share its value without being flagged.
- **Without folding:** 24 of 33 parts and 23 of 32 rows matched. The nine unmatched parts are named.
- **The negative control:** `Camberwell Green` is reported unmatched, with a warning naming it. The other 32 parts are
  unchanged.

Before the rerun, this check counted shared polygons, where Fieldwork has counted units since release 0.3.1. In
Fieldwork 0.200.0 it disagreed on the Streatham pair alone. The data now state the rule, and the check counts units.

Running the cell above repeats the check on the same data.

## What the folding bought

Plain matching is what shows the folding is worth anything. Nine of the 32 rows match only once punctuation is
folded. For example, the table writes `St. George, Camberwell`. The boundary file writes `St. George  Camberwell`,
with a doubled space and no comma.

A join on plain names would have drawn a map missing **28% of its sub-districts**. Every one of them has a comma in
its name.

The cell below lists each pair. It confirms the two names differ *only* in characters that are not letters or digits.

In [ ]:
g = report["foldingGain"]
print(f"raw {g['rawMatchedRows']} of 32 rows  ->  folded {g['foldedMatchedRows']} of 32"
      f"   ({g['rowsTheFoldingWon']} rows, {g['shareOfRows']:.0%} of the table)\n")
for p in g["pairs"]:
    mark = "OK " if p["differsOnlyInPunctuation"] else "DIFF"
    print(f"  {mark} boundary {p['boundary']!r}")
    print(f"       table {p['row']!r}")
print(f"\n{g['note']}")

f = report["folding"]
print(f"\nfolding touched nothing else: {f['agrees']}  ({f['partsChecked']} parts)")
print("criterion:", f["criterion"])
for problem in f["problems"]:
    print("  PROBLEM", problem)


## The negative control, and what the refusal is worth

The renamed part, `Camberwell Green`, scores 0.925 against `Camberwell` with both rapidfuzz and jellyfish. A matcher
that accepts names scoring 0.90 or more, a common threshold, would have **accepted** it. This widget has no threshold
and no review step. It reports the part unmatched. The two scores are what that choice is worth.

In [ ]:
n = report["negativeControl"]
print("renamed    :", f"{n['renamedFrom']!r} -> {n['renamedTo']!r}")
print("constructed:", n["constructed"])
print("unmatched  :", n["unmatched"])
print()
for name, value in n["similarityToTheNameItReplaced"].items():
    print(f"  {name:24} {value}")
print()
print(n["whatTheRefusalIsWorth"])
print()
for name, ok in n["checks"].items():
    print(f"  {'OK ' if ok else 'DIFF'} {name}")


## What this does not show

- **That folding punctuation is the right rule for names.** It is right for these two files, where every difference
  is a comma or a doubled space. A boundary spelling `St Mary` against a table's `Saint Mary` would fail this join.
  Whether failing is right there is a judgement about the data, not a result of this check. The practitioner then
  declares the two names the same, or fixes the source.
- **That the folding itself is written correctly.** It is not written again here, on purpose: a second copy by the
  same author is not a second opinion. What is checked is the rule it must obey.
- **That the 33 parts are the right geography.** Streatham being two polygons is a fact about how the map was drawn.
  Both parts take the same rate. That is correct for a rate, and would be wrong for a count.
- **That a sub-district rate measures an exposure.** In the middle 16 sub-districts, both companies supplied the same
  streets. So the rate there mixes two exposures. That is Snow's own confounding, and why he went house to house.

The cell below prints these limits as the check records them, with where the boundaries came from and their licence.

In [ ]:
for item in report["notEstablished"]:
    print("-", item)
print()
b = report["fixture"]["boundaryProvenance"]
print("boundary :", b["source"])
print("licence  :", b["licence"], "·", b["licenceSource"])
print("fixture  :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
print("pandas   :", report["environment"]["pandas"])
print("runtime  :", report["environment"]["runtime"])
